# 01 · Get the data

Downloads every raw series and saves a copy in `data/raw/`.

- 14 currency pairs and the S&P 500 from Yahoo Finance
- 3-month and 10-year interest rates for seven currencies, and the VIX, from FRED

At the end, the date of the download is saved, because Yahoo sometimes revises past data.

In [1]:
import os
from datetime import date

import pandas as pd
import yfinance as yf

import helpers

os.makedirs(helpers.RAW, exist_ok=True)

## 1. Currency pairs

In [2]:
for p in helpers.ALL_PAIRS:
    df = yf.download(p + "=X", start=helpers.START, progress=False)
    if len(df) == 0:
        print(p, "NO DATA")
    else:
        df.to_csv(f"{helpers.RAW}/{p}.csv")
        print(p, len(df), "rows,", df.index.min().date(), "to", df.index.max().date())

CADJPY 5690 rows, 2005-01-03 to 2026-11-13
AUDUSD 5333 rows, 2006-05-16 to 2026-11-14
USDJPY 5672 rows, 2005-01-03 to 2026-11-13
EURUSD 5672 rows, 2005-01-03 to 2026-11-13
EURCHF 5673 rows, 2005-01-03 to 2026-11-13
GBPJPY 5688 rows, 2005-01-03 to 2026-11-13
USDCAD 5688 rows, 2005-01-03 to 2026-11-14
GBPUSD 5684 rows, 2005-01-03 to 2026-11-13
EURGBP 5692 rows, 2005-01-03 to 2026-11-13
GBPCHF 5688 rows, 2005-01-03 to 2026-11-14
CHFJPY 5690 rows, 2005-01-03 to 2026-11-13
AUDJPY 5691 rows, 2005-01-03 to 2026-11-13
EURJPY 5675 rows, 2005-01-03 to 2026-11-13
USDCHF 5686 rows, 2005-01-03 to 2026-11-14


## 2. S&P 500 (used only to find equity crashes)

In [3]:
sp = yf.download("^GSPC", start=helpers.START, progress=False)
sp.to_csv(f"{helpers.RAW}/SP500.csv")
print("SP500", len(sp), "rows,", sp.index.min().date(), "to", sp.index.max().date())

SP500 5502 rows, 2005-01-03 to 2026-11-13


## 3. FRED series

In [4]:
for name, code in helpers.FRED_SERIES.items():
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={code}"
    df = pd.read_csv(url, index_col=0, parse_dates=True)
    df.to_csv(f"{helpers.RAW}/FRED_{name}.csv")
    values = pd.to_numeric(df.iloc[:, 0], errors="coerce").dropna()
    print(name, code, values.index.min().date(), "to", values.index.max().date())

USD_3M IR3TIB01USM156N 1964-06-01 to 2026-09-01
USD_10Y IRLTLT01USM156N 1953-04-01 to 2026-09-01
EUR_3M IR3TIB01EZM156N 1994-01-01 to 2026-01-01
EUR_10Y IRLTLT01EZM156N 1970-01-01 to 2026-01-01
GBP_3M IR3TIB01GBM156N 1957-01-01 to 2026-01-01
GBP_10Y IRLTLT01GBM156N 1960-01-01 to 2026-09-01
JPY_3M IR3TIB01JPM156N 2002-04-01 to 2026-08-01
JPY_10Y IRLTLT01JPM156N 1989-01-01 to 2026-09-01
CHF_3M IR3TIB01CHM156N 1999-07-01 to 2026-09-01
CHF_10Y IRLTLT01CHM156N 1955-01-01 to 2026-09-01
AUD_3M IR3TIB01AUM156N 1968-01-01 to 2026-08-01
AUD_10Y IRLTLT01AUM156N 1969-07-01 to 2026-09-01
CAD_3M IR3TIB01CAM156N 1956-01-01 to 2026-09-01
CAD_10Y IRLTLT01CAM156N 1955-01-01 to 2026-09-01
VIX VIXCLS 1990-01-02 to 2026-11-12


## 4. Record the download date

In [5]:
with open(f"{helpers.RAW}/download_date.txt", "w") as f:
    f.write(str(date.today()))

print("Downloaded on", date.today())

Downloaded on 2026-11-14


## 5. Checks

In [6]:
files = os.listdir(helpers.RAW)
expected = len(helpers.ALL_PAIRS) + 1 + len(helpers.FRED_SERIES) + 1
print(len(files), "files in data/raw, expected", expected)
assert len(files) == expected, "Some files are missing"

31 files in data/raw, expected 31
